# Full-issue face-audit attribute profile

Generate the Chapter 2 table comparing blind visual-label distributions for human-identified faces missed and found by the historical Faces Dataset detector. The notebook consumes only the disclosure-safe public artifacts and makes no model, API, or network calls.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ANALYSIS_ROOT = next(
    path for path in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents)
    if (path / "code/scripts/thesis_tables.py").is_file()
    and (path / "artifacts/full-issue-face-audit").is_dir()
)
ARTIFACT_DIR = ANALYSIS_ROOT / "artifacts/full-issue-face-audit/raw"
OUTPUT_DIR = ANALYSIS_ROOT / "code/output/tables"

sys.path.insert(0, str(ANALYSIS_ROOT / "code/scripts"))
from thesis_tables import export_quarto_table

## Load and validate the public blind join

The raw annotation file retains the original opaque IDs and visual labels. The separate minimal index adds detector status only after annotation. Assertions stop the export if either file is incomplete, duplicated, invalid, or unmapped.

In [ ]:
annotations = pd.DataFrame([
    json.loads(line)
    for line in (ARTIFACT_DIR / "direct-visual-annotations.jsonl")
        .read_text(encoding="utf-8").splitlines()
    if line.strip()
])
status_index = pd.read_csv(
    ARTIFACT_DIR / "blind-detection-status.csv",
    dtype={"blind_id": "string", "detection_status": "string"},
)

allowed = {
    "analyzability": ["no expression can be inferred", "low", "medium", "high"],
    "gender": ["male", "female", "ambiguous", "don't know"],
    "age": ["non-adult", "young adult", "middle adult", "old adult", "don't know"],
    "smile_presence": ["yes", "no", "don't know"],
}
assert len(annotations) == annotations["blind_id"].nunique() == 1_599
assert len(status_index) == status_index["blind_id"].nunique() == 1_599
assert set(annotations["blind_id"]) == set(status_index["blind_id"])
assert set(status_index["detection_status"]) == {"missed", "found"}
for field, values in allowed.items():
    assert set(annotations[field]).issubset(values), field

joined = annotations.merge(
    status_index,
    on="blind_id",
    how="inner",
    validate="one_to_one",
)
assert joined["detection_status"].eq("missed").sum() == 1_140
assert joined["detection_status"].eq("found").sum() == 459

display(pd.Series({
    "Joined faces": len(joined),
    "Unique blind IDs": joined["blind_id"].nunique(),
    "Detector-missed": joined["detection_status"].eq("missed").sum(),
    "Detector-found": joined["detection_status"].eq("found").sum(),
}).to_frame("n"))

## Construct and export the Chapter 2 table

Each cell reports the count and share within detector-missed or detector-found faces.

In [ ]:
field_labels = {
    "analyzability": "Expression analyzability",
    "gender": "Gender presentation",
    "age": "Perceived age",
    "smile_presence": "Smile presence",
}
value_labels = {
    "no expression can be inferred": "No expression inferable",
    "low": "Low", "medium": "Medium", "high": "High",
    "male": "Male", "female": "Female", "ambiguous": "Ambiguous",
    "don't know": "Don't know",
    "non-adult": "Non-adult", "young adult": "Young adult",
    "middle adult": "Middle adult", "old adult": "Old adult",
    "yes": "Yes", "no": "No",
}
denominators = {
    status: int(joined["detection_status"].eq(status).sum())
    for status in ("missed", "found")
}

def count_share(mask: pd.Series, denominator: int) -> str:
    count = int(mask.sum())
    return f"{count:,} ({count / denominator:.1%})"

rows = []
for field, order in allowed.items():
    for value in order:
        category = joined[field].eq(value)
        rows.append({
            "attribute": field_labels[field],
            "label": value_labels[value],
            "missed": count_share(
                category & joined["detection_status"].eq("missed"),
                denominators["missed"],
            ),
            "found": count_share(
                category & joined["detection_status"].eq("found"),
                denominators["found"],
            ),
        })
table = pd.DataFrame(rows)

values_path, fragment_path = export_quarto_table(
    table,
    "full-issue-face-audit-attribute-profile",
    caption=(
        "Blind visual-label composition of faces missed and found by the "
        "historical Faces Dataset detector."
    ),
    label="tbl-full-issue-face-audit-attribute-profile",
    column_labels={
        "attribute": "Attribute",
        "label": "Label",
        "missed": "Missed (n=1,140)",
        "found": "Found (n=459)",
    },
    alignments={"missed": "right", "found": "right"},
    note=(
        "Cells report count (within-column share). Labels were assigned blindly "
        "from extended face crops and are perceived attributes rather than "
        "demographic ground truth."
    ),
    data_dir=OUTPUT_DIR,
    qmd_dir=OUTPUT_DIR,
)

display(table)
print(f"Aggregate values: {values_path}")
print(f"Quarto fragment: {fragment_path}")